# Lasso回帰によるMEDV予測

第2回の宿題コードから最小限の変更で、`boston.csv` の `MEDV` をLasso回帰で予測するコードです。

【Python セミナー 第3回】  
第2回のXGBoostによる予測コードを、Lasso回帰によるMEDV予測に変更する。

In [ ]:
# 必要なライブラリを読み込む。
# ライブラリとは、よく使う機能をまとめた道具箱のようなもの。

# 【NumPy】numpy は数値計算を効率よく行うためのライブラリ。
# ここでは NumPy の np.sqrt() や np.linspace() などを使う。
# import numpy as np と書くことで、以降は numpy を np という短い名前で呼び出せる。
import numpy as np

# 【pandas】pandas は表形式のデータを扱うためのライブラリ。
# CSV ファイルを読み込む pd.read_csv()、表を作る pd.DataFrame()、列を消す df.drop() などで使う。
# import pandas as pd と書くことで、以降は pandas を pd という短い名前で呼び出せる。
import pandas as pd

# 【matplotlib】matplotlib はグラフを描くための基本的なライブラリ。
# ここでは matplotlib.pyplot の plt.subplots(), plt.figure(), plt.show() などを使う。
# import matplotlib.pyplot as plt と書くことで、以降は pyplot を plt という短い名前で呼び出せる。
import matplotlib.pyplot as plt

# 【scikit-learn】scikit-learn は機械学習でよく使うライブラリ。
# train_test_split(), mean_squared_error(), mean_absolute_error(), r2_score() などが scikit-learn 由来。
# データ分割、評価指標、回帰モデルの作成などに使う。
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 【scikit-learn】LassoCV は交差検証で alpha を自動的に選ぶ Lasso 回帰モデル。
# 係数の一部が0になりやすいため、特徴量選択にも使える。
from sklearn.linear_model import LassoCV

: 

In [ ]:
# 予測結果を評価し、実測値と予測値の散布図を描く関数を定義する。
# def は「関数を作る」という意味。
# 関数にしておくと、同じ処理を何度も書かずに使い回せる。
def pred_plot(train_targets, train_predictions, test_targets, test_predictions, model_name):
    # train_targets: トレーニングデータの正解値
    # train_predictions: トレーニングデータに対する予測値
    # test_targets: テストデータの正解値
    # test_predictions: テストデータに対する予測値
    # model_name: グラフタイトルに表示するモデル名

    # まず、トレーニングデータに対する予測の良さを計算する。
    # 【scikit-learn】mean_squared_error() は sklearn.metrics の関数。
    # MSE は平均二乗誤差。誤差を二乗して平均した値で、小さいほど良い。
    train_mse = mean_squared_error(train_targets, train_predictions)

    # 【scikit-learn】mean_absolute_error() は sklearn.metrics の関数。
    # MAE は平均絶対誤差。実測値と予測値の差の絶対値を平均した値。
    train_mae = mean_absolute_error(train_targets, train_predictions)

    # 【scikit-learn】r2_score() は sklearn.metrics の関数。
    # R2 は決定係数。1 に近いほどよく予測できていることを表す。
    train_r2 = r2_score(train_targets, train_predictions)

    # 【NumPy】np.sqrt() は NumPy の関数で、平方根を計算する。
    # RMSE は MSE の平方根。目的変数と同じ単位で誤差を見られる。
    train_rmse = np.sqrt(train_mse)

    # 次に、未知データであるテストデータに対する予測の良さを計算する。
    test_mse = mean_squared_error(test_targets, test_predictions)
    test_mae = mean_absolute_error(test_targets, test_predictions)
    test_r2 = r2_score(test_targets, test_predictions)
    test_rmse = np.sqrt(test_mse)

    # グラフを描くために、実測値と予測値を1次元の配列にそろえる。
    # np.asarray() は、リストや pandas のデータを NumPy 配列に変換する。
    # .ravel() は、(行数, 1) のような2次元配列を、(行数,) の1次元配列に変換する。
    # scatter() では x 軸と y 軸に同じ長さの1次元配列を渡すと扱いやすい。
    train_targets_array = np.asarray(train_targets).ravel()
    train_predictions_array = np.asarray(train_predictions).ravel()
    test_targets_array = np.asarray(test_targets).ravel()
    test_predictions_array = np.asarray(test_predictions).ravel()

    # f-string を使うと、文字列の中に変数の値を埋め込める。
    # :.4f は「小数点以下 4 桁まで表示する」という意味。
    print(f'\n{model_name} Training Set Performance:')
    print(f'Samples: {len(train_predictions)} | RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f} | R2: {train_r2:.4f}')

    print(f'\n{model_name} Test Set Performance:')
    print(f'Samples: {len(test_predictions)} | RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f} | R2: {test_r2:.4f}')

    # 【matplotlib】plt.subplots() は matplotlib.pyplot の関数。
    # 横に 2 つ並んだグラフ領域を作る。
    # fig は図全体、ax1 と ax2 はそれぞれ左側・右側のグラフ。
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    # 【matplotlib】ax1.scatter() は matplotlib の Axes オブジェクトのメソッドで、散布図を描く。
    # 左側のグラフ: トレーニングデータの実測値と予測値を散布図で表示する。
    ax1.scatter(train_targets_array, train_predictions_array, alpha=0.6, color='red', s=20)

    # y = x の基準線を描く。
    # 点がこの線に近いほど、予測値が実測値に近いという意味。
    # 【NumPy】np.linspace() は、指定した範囲を等間隔に分けた数値列を作る関数。
    train_min = min(train_targets_array.min(), train_predictions_array.min())
    train_max = max(train_targets_array.max(), train_predictions_array.max())
    train_range = np.linspace(train_min, train_max)
    ax1.plot(train_range, train_range, 'k--', lw=2)

    # 軸ラベルやタイトルを設定する。
    ax1.set_xlabel('Actual MEDV')
    ax1.set_ylabel('Predicted MEDV')
    ax1.set_title(f'{model_name} Training Set (n={len(train_predictions)})\nRMSE: {train_rmse:.3f} | MAE: {train_mae:.3f} | R2: {train_r2:.3f}')
    ax1.grid(True, alpha=0.3)

    # x 軸と y 軸の縮尺を同じにして、y = x の線を見やすくする。
    ax1.set_aspect('equal', adjustable='box')

    # 右側のグラフ: テストデータの実測値と予測値を散布図で表示する。
    ax2.scatter(test_targets_array, test_predictions_array, alpha=0.6, color='blue', s=20)
    test_min = min(test_targets_array.min(), test_predictions_array.min())
    test_max = max(test_targets_array.max(), test_predictions_array.max())
    test_range = np.linspace(test_min, test_max)
    ax2.plot(test_range, test_range, 'k--', lw=2)
    ax2.set_xlabel('Actual MEDV')
    ax2.set_ylabel('Predicted MEDV')
    ax2.set_title(f'{model_name} Test Set (n={len(test_predictions)})\nRMSE: {test_rmse:.3f} | MAE: {test_mae:.3f} | R2: {test_r2:.3f}')
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal', adjustable='box')

    # 【matplotlib】plt.tight_layout() は、グラフ同士や文字が重ならないように余白を自動調整する。
    plt.tight_layout()

    # 【matplotlib】plt.show() は、作成したグラフを画面に表示する関数。
    plt.show()

In [ ]:
# Lasso回帰で得られた係数を可視化する関数。
def lasso_coefficients(model, X, model_name):
    # 【scikit-learn】Lasso の coef_ には、各説明変数に対応する係数が入っている。
    # 係数が 0 の特徴量は、Lassoによって使われなかった特徴量と考えられる。
    coefficients = model.coef_

    # X は pandas の DataFrame なので、columns で列名を取得できる。
    feature_names = X.columns

    # 特徴量名と係数を 1 つの表にまとめる。
    coefficient_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})

    # 係数の絶対値が大きい順に並べ替える。
    coefficient_df['AbsCoefficient'] = coefficient_df['Coefficient'].abs()
    coefficient_df = coefficient_df.sort_values(by='AbsCoefficient', ascending=False)

    # 係数の一覧を表示する。
    display(coefficient_df[['Feature', 'Coefficient']])

    # 係数が 0 ではない特徴量だけを棒グラフで表示する。
    selected_df = coefficient_df[coefficient_df['AbsCoefficient'] > 0]

    plt.figure(figsize=(10, 6))
    plt.barh(selected_df['Feature'], selected_df['Coefficient'])
    plt.xlabel('Coefficient')
    plt.ylabel('Feature')
    plt.title(f'Coefficients in {model_name}')
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3)
    plt.show()

    print(f'使用された特徴量数: {selected_df.shape[0]} / {X.shape[1]}')

In [ ]:
# CSV ファイルを読み込む。
# CSV はカンマ区切りの表形式データ。
# 【pandas】pd.read_csv() を使うと、CSV を pandas の DataFrame として読み込める。
# index_col=0 は、CSVの1列目を行名として使うという意味。
df = pd.read_csv('boston.csv', index_col=0)

# Jupyter Notebook では、セルの最後に変数名を書くと中身が表示される。
# ここでは読み込んだ表 df を確認している。
df

In [ ]:
# Boston housing データでは、全ての列が数値データとして使える。
# 第2回コードの drop_list と df_clean の形を残すため、ここでは削除する列を空のリストにする。
drop_list = []

# 【pandas】drop(columns=...) は、指定した列を DataFrame から削除する pandas のメソッド。
# 今回は削除する列がないため、df と同じ内容が df_clean に入る。
df_clean = df.drop(columns=drop_list)

# データを確認する。
df_clean

In [ ]:
# 予測したい列を目的変数として指定する。
# ここでは MEDV という列を予測対象にする。
target = 'MEDV'

# y は目的変数。
# 機械学習では、予測したい値を y と書くことがよくある。
y = df_clean[target]

# x は説明変数。
# 目的変数 MEDV 以外の列を、予測に使う入力データとして使う。
x = df_clean.drop(columns=[target])

# 【scikit-learn】train_test_split() は sklearn.model_selection の関数。
# データをトレーニングセットとテストセットに分ける。
# トレーニングセットはモデルの学習に使い、テストセットは学習後の性能確認に使う。
# test_size=0.20 は、全体の 20% をテストデータにするという意味。
# random_state=42 を指定すると、毎回同じ分け方になり、結果を再現しやすくなる。
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.20, shuffle=True, random_state=42)

# 目的変数 y_train を標準化する。
# 標準化とは「平均を 0、標準偏差を 1 に近づける変換」。
# 値のスケールをそろえることで、モデルが学習しやすくなる場合がある。
auto_y_train = (y_train - y_train.mean()) / y_train.std()

# 説明変数 X_train も同じように標準化する。
# 各列ごとに、トレーニングデータの平均と標準偏差を使って変換する。
auto_X_train = (X_train - X_train.mean()) / X_train.std()

# テストデータも標準化する。
# ここで重要なのは、テストデータ自身の平均・標準偏差ではなく、
# トレーニングデータ X_train の平均・標準偏差を使うこと。
# これは、未知データを予測する状況を正しく再現するため。
auto_X_test = (X_test - X_train.mean()) / X_train.std()

# データの概要を表示する。
# x.shape[1] は列数、x.shape[0] は行数。
print(f'目的変数:{target},説明変数:{x.shape[1]},データ数:{x.shape[0]}')
print(f'トレーニングセット:{X_train.shape[0]},テストセット:{X_test.shape[0]}')

In [ ]:
# Lasso回帰を使って回帰モデルを作る。
# 回帰とは、連続値を予測する問題。
# 今回は住宅価格の中央値を表す MEDV を予測する。

# グラフタイトルなどで使うモデル名を文字列として保存する。
model_name = 'Lasso'

# 【scikit-learn】LassoCV は複数の alpha を試し、交差検証で最も良い alpha を自動的に選ぶ。
# alpha は正則化の強さを表す。大きくすると係数が 0 になる特徴量が増えやすい。
# cv=5 はトレーニングデータを 5 分割して検証するという意味。
# alphas=100 は、データに合わせて 100 個の alpha 候補を自動生成して試すという意味。
model = LassoCV(alphas=100, cv=5, max_iter=100000, random_state=42)

# 【scikit-learn 形式】fit() はモデルを学習させる命令。
# auto_X_train を入力、auto_y_train を正解として、入力と正解の関係を学習する。
model.fit(auto_X_train, auto_y_train)

# LassoCV が選んだ最適な alpha を確認する。
print(f'最適な alpha: {model.alpha_:.6f}')

# 【scikit-learn 形式】predict() は、学習済みモデルで予測する命令。
# 学習済みモデルを使って、トレーニングデータの y を予測する。
# 予測値は標準化されたスケールで出てくる。
autoscaled_pred_y_train = model.predict(auto_X_train)

# 標準化された予測値を、元の MEDV のスケールに戻す。
# 標準化の逆変換は「標準偏差をかけて、平均を足す」。
pred_y_train = autoscaled_pred_y_train * y_train.std() + y_train.mean()

# 【pandas】pd.DataFrame() を使って、予測値を pandas の DataFrame に変換する。
# index を元データとそろえることで、どの行の予測か分かりやすくなる。
pred_y_train = pd.DataFrame(pred_y_train, index=auto_X_train.index, columns=['pred_y'])

# テストデータについても同じように予測する。
autoscaled_pred_y_test = model.predict(auto_X_test)

# テストデータの予測値も、トレーニングデータの y_train の平均・標準偏差を使って元のスケールに戻す。
pred_y_test = autoscaled_pred_y_test * y_train.std() + y_train.mean()
pred_y_test = pd.DataFrame(pred_y_test, index=X_test.index, columns=['pred_y'])

# 実測値と予測値の比較グラフ、RMSE・MAE・R2 などの評価指標を表示する。
pred_plot(y_train, pred_y_train, y_test, pred_y_test, model_name)

# Lasso回帰で得られた係数を棒グラフで表示する。
lasso_coefficients(model, auto_X_train, model_name)

In [ ]:
# 予測値の一部を表で確認する。
# actual_MEDV が実測値、predicted_MEDV がLasso回帰による予測値。
prediction_sample = pd.DataFrame(
    {
        'actual_MEDV': y_test,
        'predicted_MEDV': pred_y_test['pred_y'],
    }
)

prediction_sample.head(10)